# Gemini smoke test

This notebook loads environment variables from `.env`, initializes the Vertex AI Gemini client, and generates a simple "hello world" response.


In [6]:
from dotenv import load_dotenv
from google.oauth2 import service_account
from google.auth.transport.requests import Request
from datetime import datetime, timezone
import os, json, requests

# 1) Load env
load_dotenv(dotenv_path="/home/clding/continual-learning/src/lm/.env", override=True)
creds = os.getenv("GCP_CREDENTIALS", "/home/clding/continual-learning/src/gcp.json")
project = os.getenv("GCP_PROJECT", "")
region = os.getenv("GCP_REGION", "us-central1")

# 2) Who is calling?
with open(os.path.expanduser(creds), "r") as f:
    sa = json.load(f)
print("Service Account:", sa.get("client_email"))
print("SA project_id  :", sa.get("project_id"))
print("Env project    :", repr(project))
print("Env region     :", repr(region))

# 3) Mint OAuth2 access token
scopes = [
    "https://www.googleapis.com/auth/cloud-platform",
]
cred = service_account.Credentials.from_service_account_file(os.path.expanduser(creds), scopes=scopes)
cred.refresh(Request())

expiry = cred.expiry
if expiry.tzinfo is None:
    expiry = expiry.replace(tzinfo=timezone.utc)
print("Now (UTC)      :", datetime.now(timezone.utc))
print("Token expiry   :", expiry)
print("Secs to expiry :", (expiry - datetime.now(timezone.utc)).total_seconds())

# 4) Auth check against Google API (projects.get)
headers = {"Authorization": f"Bearer {cred.token}"}
url = f"https://cloudresourcemanager.googleapis.com/v1/projects/{project}"
r = requests.get(url, headers=headers, timeout=30)
print("projects.get ->", r.status_code)
print(r.text[:400])

# Interpret:
# - 200: Auth OK and you can read the project (permissions OK for this call)
# - 403: Auth OK but the caller lacks permissions on this project (common, not an auth failure)
# - 401 / invalid_grant: Auth NOT OK (clock skew, bad key, or wrong token type)

Service Account: bird-tester@hailongli-playground2.iam.gserviceaccount.com
SA project_id  : hailongli-playground2
Env project    : 'sercan-v1'
Env region     : 'us-central1'
Now (UTC)      : 2025-10-25 11:45:11.339749+00:00
Token expiry   : 2025-10-25 12:45:10.338856+00:00
Secs to expiry : 3598.998975
projects.get -> 200
{
  "projectNumber": "618488765595",
  "projectId": "sercan-v1",
  "lifecycleState": "ACTIVE",
  "name": "sercan-v1",
  "labels": {
    "generative-language": "enabled"
  },
  "createTime": "2018-10-12T23:15:13.720Z",
  "parent": {
    "type": "folder",
    "id": "709605854314"
  }
}



In [1]:
# Load .env and validate required env vars
from dotenv import load_dotenv
import os, sys, json, pathlib

# Load .env from src/lm explicitly
load_dotenv(dotenv_path="/home/clding/continual-learning/src/lm/.env", override=True)

creds = os.getenv("GCP_CREDENTIALS")
project = os.getenv("GCP_PROJECT")
region = os.getenv("GCP_REGION")
model = os.getenv("GEMINI_MODEL", "gemini-1.5-flash-002")

# Expand ~ and env vars in path
if creds:
    creds = os.path.expanduser(os.path.expandvars(creds))

print("Resolved env:")
print(" GCP_CREDENTIALS:", creds)
print(" GCP_PROJECT:", project)
print(" GCP_REGION:", region)
print(" GEMINI_MODEL:", model)

assert creds and pathlib.Path(creds).exists(), f"Credentials file not found: {creds}"
assert project, "GCP_PROJECT not set"
assert region, "GCP_REGION not set"


Resolved env:
 GCP_CREDENTIALS: /home/clding/continual-learning/src/gcp.json
 GCP_PROJECT: sercan-v1
 GCP_REGION: us-central1
 GEMINI_MODEL: gemini-1.5-flash-002


In [23]:
# Initialize Gemini client
from google import genai
from google.oauth2 import service_account
from google.genai.types import GenerateContentConfig

credentials = service_account.Credentials.from_service_account_file(
    creds,
    scopes=[
        "https://www.googleapis.com/auth/generative-language",
        "https://www.googleapis.com/auth/cloud-platform",
    ],
)

client = genai.Client(
    vertexai=True,
    project=project,
    location=region,
    credentials=credentials,
)

print("Client initialized.")


Client initialized.


In [24]:
import os, json, requests
from google.oauth2 import service_account
from google.auth.transport.requests import Request

# Inputs
creds_path = os.path.expanduser(os.getenv("GCP_CREDENTIALS", "/home/clding/continual-learning/src/gcp.json"))
project_env = os.getenv("GCP_PROJECT")
region_env = os.getenv("GCP_REGION", "us-central1")

# Load SA JSON
with open(creds_path, "r") as f:
    sa = json.load(f)

print("SA email:", sa.get("client_email"))
print("SA project_id (from key):", sa.get("project_id"))
print("Target project (env):", project_env)
print("Region:", region_env)
print("Cross-project?", sa.get("project_id") != project_env)

# Build access token
scopes = [
    "https://www.googleapis.com/auth/generative-language",
    "https://www.googleapis.com/auth/cloud-platform",
]
credentials = service_account.Credentials.from_service_account_file(creds_path, scopes=scopes)
credentials.refresh(Request())
token = credentials.token

def list_publisher_models(project: str, region: str):
    url = f"https://{region}-aiplatform.googleapis.com/v1/projects/{project}/locations/{region}/publishers/google/models"
    r = requests.get(url, headers={"Authorization": f"Bearer {token}"}, timeout=30)
    print(f"LIST {region} status:", r.status_code)
    if r.status_code == 200:
        names = [m.get("name") for m in r.json().get("models", []) or []]
        print(f"Models visible in {region}:", len(names))
        print(*(names[:10]), sep="\n")  # show a few
    else:
        print("LIST body:", r.text[:500])

for reg in [region_env, "us-central1", "europe-west4"]:
    list_publisher_models(project_env, reg)

def try_generate(project: str, region: str, model_id: str):
    url = f"https://{region}-aiplatform.googleapis.com/v1/projects/{project}/locations/{region}/publishers/google/models/{model_id}:generateContent"
    payload = {
        "contents": [{"role": "user", "parts": [{"text": "hello world"}]}]
    }
    r = requests.post(url, headers={
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }, json=payload, timeout=60)
    print(f"GEN {region} {model_id} status:", r.status_code)
    print("GEN body:", r.text[:500])

for reg in [region_env, "us-central1", "europe-west4"]:
    for mid in ["gemini-1.5-flash-002","gemini-1.5-pro-002"]:
        try_generate(project_env, reg, mid)

SA email: bird-tester@hailongli-playground2.iam.gserviceaccount.com
SA project_id (from key): hailongli-playground2
Target project (env): hailongli-playground2
Region: us-central1
Cross-project? False
LIST us-central1 status: 404
LIST body: <!DOCTYPE html>
<html lang=en>
  <meta charset=utf-8>
  <meta name=viewport content="initial-scale=1, minimum-scale=1, width=device-width">
  <title>Error 404 (Not Found)!!1</title>
  <style>
    *{margin:0;padding:0}html,code{font:15px/22px arial,sans-serif}html{background:#fff;color:#222;padding:15px}body{margin:7% auto 0;max-width:390px;min-height:180px;padding:30px 0 15px}* > body{background:url(//www.google.com/images/errors/robot.png) 100% 5px no-repeat;padding-right:205px}p{margin:11px 0 
LIST us-central1 status: 404
LIST body: <!DOCTYPE html>
<html lang=en>
  <meta charset=utf-8>
  <meta name=viewport content="initial-scale=1, minimum-scale=1, width=device-width">
  <title>Error 404 (Not Found)!!1</title>
  <style>
    *{margin:0;padding:0}h

In [ ]:
# Auth/token diagnostics
from datetime import datetime, timezone
import pkg_resources, os
from google.oauth2 import service_account
from google.auth.transport.requests import Request

print("GOOGLE_APPLICATION_CREDENTIALS:", os.environ.get("GOOGLE_APPLICATION_CREDENTIALS"))
print("GCP_PROJECT:", os.environ.get("GCP_PROJECT"))
print("GCP_REGION:", os.environ.get("GCP_REGION"))
print("Now (UTC):", datetime.now(timezone.utc))

print("lib versions:")
for lib in ["google-auth","google-api-core","google-genai","google-cloud-aiplatform"]:
    try:
        print(lib, pkg_resources.get_distribution(lib).version)
    except Exception:
        pass

cred = service_account.Credentials.from_service_account_file(
    creds,
    scopes=[
        "https://www.googleapis.com/auth/generative-language",
        "https://www.googleapis.com/auth/cloud-platform",
    ],
)
cred.refresh(Request())
print("Access token expiry:", cred.expiry)
print("Seconds to expiry:", (cred.expiry - datetime.now(timezone.utc)).total_seconds())


In [8]:
# Try loading and calling a Gemini model (client + REST)
from google import genai
from google.genai.errors import ClientError
from google.genai.types import GenerateContentConfig

# Reuse env vars from previous cell: creds, project, region
# Choose model from env or default
model = os.getenv("GEMINI_MODEL", "gemini-2.5-flash-lite")
model_fq = f"projects/{project}/locations/{region}/publishers/google/models/{model}"
print("Model short:", model)
print("Model FQN  :", model_fq)

# 1) Client call
try:
    client = genai.Client(vertexai=True, project=project, location=region, credentials=cred)
    cfg = GenerateContentConfig(
        system_instruction=None,
        temperature=float(os.getenv("GEMINI_TEMPERATURE", 0.2)),
        max_output_tokens=int(os.getenv("GEMINI_MAX_TOKENS", 128)),
    )
    resp = client.models.generate_content(
        model=model_fq,
        contents=[{"role": "user", "parts": [{"text": "hello world"}]}],
        config=cfg,
    )
    print("Client OK. Text:\n", resp.text)
except Exception as e:
    print("Client FAIL:", e)

# 2) REST call with the same FQN
try:
    url = f"https://{region}-aiplatform.googleapis.com/v1/{model_fq}:generateContent"
    payload = {"contents": [{"role": "user", "parts": [{"text": "hello world"}]}]}
    r = requests.post(url, headers={"Authorization": f"Bearer {cred.token}", "Content-Type": "application/json"}, json=payload, timeout=60)
    print("REST status:", r.status_code)
    print(r.text[:500])
except Exception as e:
    print("REST FAIL:", e)


Model short: gemini-2.5-flash-lite
Model FQN  : projects/sercan-v1/locations/us-central1/publishers/google/models/gemini-2.5-flash-lite
Client OK. Text:
 Hello there! How can I help you today?
REST status: 200
{
  "candidates": [
    {
      "content": {
        "role": "model",
        "parts": [
          {
            "text": "Hello! How can I help you today?"
          }
        ]
      },
      "finishReason": "STOP",
      "score": -2.6679496765136719,
      "avgLogprobs": -0.29643885294596356
    }
  ],
  "usageMetadata": {
    "promptTokenCount": 2,
    "candidatesTokenCount": 9,
    "totalTokenCount": 11,
    "billablePromptUsage": {
      "textCount": 10
    },
    "trafficType": "ON_DEMAND"


In [11]:
import os, requests
project=os.getenv("GCP_PROJECT"); region=os.getenv("GCP_REGION","us-central1")
url=f"https://{region}-aiplatform.googleapis.com/v1/projects/{project}/locations/{region}/publishers/google/models"
r=requests.get(url, headers={"Authorization": f"Bearer {cred.token}"}, timeout=30)
print(r.status_code, r.text[:1000])

404 <!DOCTYPE html>
<html lang=en>
  <meta charset=utf-8>
  <meta name=viewport content="initial-scale=1, minimum-scale=1, width=device-width">
  <title>Error 404 (Not Found)!!1</title>
  <style>
    *{margin:0;padding:0}html,code{font:15px/22px arial,sans-serif}html{background:#fff;color:#222;padding:15px}body{margin:7% auto 0;max-width:390px;min-height:180px;padding:30px 0 15px}* > body{background:url(//www.google.com/images/errors/robot.png) 100% 5px no-repeat;padding-right:205px}p{margin:11px 0 22px;overflow:hidden}ins{color:#777;text-decoration:none}a img{border:0}@media screen and (max-width:772px){body{background:none;margin-top:0;max-width:none;padding-right:0}}#logo{background:url(//www.google.com/images/branding/googlelogo/1x/googlelogo_color_150x54dp.png) no-repeat;margin-left:-5px}@media only screen and (min-resolution:192dpi){#logo{background:url(//www.google.com/images/branding/googlelogo/2x/googlelogo_color_150x54dp.png) no-repeat 0% 0%/100% 100%;-moz-border-image:url(//

In [25]:
from google import genai
from google.genai.errors import ClientError

candidates = [
    "gemini-1.5-flash-002",
    "gemini-1.5-flash",
    "gemini-1.5-pro-002",
    "gemini-1.5-pro",
]
regions = [region, "us-central1", "europe-west4"]

def try_generate(reg, model_id):
    cli = genai.Client(vertexai=True, project=project, location=reg, credentials=credentials)
    # Try short name
    try:
        r = cli.models.generate_content(
            model=model_id,
            contents=[{"role": "user", "parts": [{"text": "hello world"}]}],
            config=cfg,
        )
        return ("OK", reg, model_id, r.text)
    except ClientError as e:
        # Try fully-qualified path
        try:
            fq = f"projects/{project}/locations/{reg}/publishers/google/models/{model_id}"
            r = cli.models.generate_content(
                model=fq,
                contents=[{"role": "user", "parts": [{"text": "hello world"}]}],
                config=cfg,
            )
            return ("OK", reg, fq, r.text)
        except Exception as e2:
            return ("ERR", reg, model_id, str(e2))

results = []
for reg in regions:
    for m in candidates:
        status, reg_used, model_used, info = try_generate(reg, m)
        print(status, reg_used, model_used)
        if status == "OK":
            print("Response:", info)
            raise SystemExit  # stop on first success
        else:
            print("Reason:", info[:300])

print("All attempts failed. Likely access/region/permissions issue.")

ERR us-central1 gemini-1.5-flash-002
Reason: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'Publisher Model `projects/hailongli-playground2/locations/us-central1/publishers/google/models/gemini-1.5-flash-002` was not found or your project does not have access to it. Please ensure you are using a valid model version. For more information, s
ERR us-central1 gemini-1.5-flash
Reason: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'Publisher Model `projects/hailongli-playground2/locations/us-central1/publishers/google/models/gemini-1.5-flash` was not found or your project does not have access to it. Please ensure you are using a valid model version. For more information, see: 
ERR us-central1 gemini-1.5-pro-002
Reason: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'Publisher Model `projects/hailongli-playground2/locations/us-central1/publishers/google/models/gemini-1.5-pro-002` was not found or your project does not have access to it. Please ensure you are using a valid model ver

In [10]:
[m.name for m in client.models.list() if "/publishers/google/models/gemini" in m.name]

[]

In [13]:
# Generate a simple response
cfg = GenerateContentConfig(
    system_instruction=None,
    temperature=float(os.getenv("GEMINI_TEMPERATURE", 0.2)),
    max_output_tokens=int(os.getenv("GEMINI_MAX_TOKENS", 128)),
)

resp = client.models.generate_content(
    model=model,
    contents=[{"role": "user", "parts": [{"text": "what is 1+1?"}]}],
    config=cfg,
)

print("Response text:\n", resp.text)


Response text:
 1 + 1 = 2
